# Pathogenic Interest Index

Rebuild the released index table from Phase 4 species counts. The calculation estimates the pathogen-associated publication count from sampled abstracts and normalizes it separately within `bacteria`, `viruses`, and `fungi`. Scores are comparable within a domain and are not intended for cross-domain comparison.

Run Phase 4 first so `species_summary.parquet` contains current counts. When bacteria are not part of the selected V3 run, this notebook retains the migrated bacterial cohort for species without V3 pathogen evidence and applies the same index formula. Fresh accepted V3 evidence takes precedence. The Explorer reads the resulting `pathogenic_interest_index.parquet` when it builds its public data snapshot.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pathogen_atlas_v3').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v3.common.io import load_yaml, atomic_parquet
from pathogen_atlas_v3.tables.interest_index import INDEX_VERSION, build_pathogenic_interest_index

CONFIG_PATH = PROJECT_ROOT / 'configs/default.yaml'
CONFIG = load_yaml(CONFIG_PATH)
RELEASE_DIR = PROJECT_ROOT / CONFIG['paths']['release_root']
SUMMARY_PATH = RELEASE_DIR / 'species_summary.parquet'
INDEX_PATH = RELEASE_DIR / 'pathogenic_interest_index.parquet'
print(f'Input:  {SUMMARY_PATH}')
print(f'Output: {INDEX_PATH}')
print(f'Version: {INDEX_VERSION}')


Input:  /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version3/data/04_release/species_summary.parquet
Output: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version3/data/04_release/pathogenic_interest_index.parquet
Version: pathogenic-interest-index-v3.2-sampled


In [2]:
if not SUMMARY_PATH.exists():
    raise FileNotFoundError('Run notebooks/04_release.ipynb before computing the index.')

summary = pd.read_parquet(SUMMARY_PATH)
count_columns = [
    'organism_id', 'organism_group', 'n_pathogen_pmids',
    'n_abstracts_fetched', 'n_pubmed_search_records',
]
missing = sorted(set(count_columns) - set(summary.columns))
if missing:
    raise ValueError(f'Species summary is missing index inputs: {missing}')

index_input = summary[count_columns].copy()
index_input['index_source'] = 'v3'

# Retain the established bacterial cohort while V3 bacterial acquisitions are
# incomplete. Match by accepted species name only within the bacteria domain.
v2_summary_path = PROJECT_ROOT / 'data/migration/version2_snapshot/04_tables/release/species_summary.parquet'
if v2_summary_path.exists():
    v2_summary = pd.read_parquet(v2_summary_path)
    bacteria_ids = {
        str(row.accepted_name).casefold(): str(row.organism_id)
        for row in summary.loc[summary['organism_group'].eq('bacteria')].itertuples(index=False)
    }
    v2_summary['organism_id'] = v2_summary['canonical_species'].astype(str).str.casefold().map(bacteria_ids)
    v2_summary = v2_summary.loc[v2_summary['organism_id'].notna()].copy()
    legacy_counts = v2_summary.set_index('organism_id')[['n_causation', 'n_sampled', 'n_articles']]
    legacy_counts.columns = ['n_pathogen_pmids', 'n_abstracts_fetched', 'n_pubmed_search_records']
    legacy_counts = legacy_counts.apply(pd.to_numeric, errors='coerce').fillna(0)
    legacy_ids = set(legacy_counts.index)
    selected_groups = set(map(str, CONFIG.get('run', {}).get('organism_groups', [])))
    no_v3_pathogen_evidence = pd.to_numeric(index_input['n_pathogen_pmids'], errors='coerce').fillna(0).eq(0)
    use_legacy = (
        index_input['organism_group'].eq('bacteria')
        & no_v3_pathogen_evidence
        & index_input['organism_id'].astype(str).isin(legacy_ids)
        & ('bacteria' not in selected_groups)
    )
    legacy_by_id = legacy_counts.to_dict(orient='index')
    for column in ('n_pathogen_pmids', 'n_abstracts_fetched', 'n_pubmed_search_records'):
        index_input.loc[use_legacy, column] = index_input.loc[use_legacy, 'organism_id'].astype(str).map(
            {oid: values[column] for oid, values in legacy_by_id.items()}
        )
    index_input.loc[use_legacy, 'index_source'] = 'v2_migrated_counts'

# Show coverage and count totals by domain before calculating the scores.
display(index_input.groupby('organism_group', dropna=False).agg(
    species=('organism_id', 'nunique'),
    v2_migrated_species=('index_source', lambda values: int(values.eq('v2_migrated_counts').sum())),
    accepted_pathogen_pmids=('n_pathogen_pmids', 'sum'),
    abstracts_fetched=('n_abstracts_fetched', 'sum'),
    pubmed_search_records=('n_pubmed_search_records', 'sum'),
).reset_index())


,organism_group,species,v2_migrated_species,accepted_pathogen_pmids,abstracts_fetched,pubmed_search_records
0,bacteria,22804,22799,27031,161471,1498296
1,fungi,101843,0,0,0,0
2,viruses,17554,0,5741,187325,1908216


In [3]:
index = build_pathogenic_interest_index(index_input[count_columns])
index = index.merge(index_input[['organism_id', 'index_source']], on='organism_id', how='left', validate='one_to_one')

# Confirm that normalization is independent within each organism domain.
domain_check = index.groupby('organism_group', dropna=False).agg(
    species=('organism_id', 'nunique'),
    nonzero_scores=('pathogenic_interest_index', lambda values: int(values.gt(0).sum())),
    maximum_index=('pathogenic_interest_index', 'max'),
).reset_index()
display(domain_check)
display(index.sort_values(
    ['organism_group', 'pathogenic_interest_index'], ascending=[True, False]
).groupby('organism_group', sort=False).head(10))


,organism_group,species,nonzero_scores,maximum_index
0,bacteria,22804,1172,1.0
1,fungi,101843,0,0.0
2,viruses,17554,227,1.0


,organism_id,organism_group,n_pathogen_pmids,n_abstracts_fetched,n_pubmed_search_records,estimated_pathogen_pmids,log_estimated_pathogen_pmids,pathogenic_interest_index,index_version,index_source
197,bacteria:02ecf4958d9c70114c23,bacteria,108,200,16119,8704.260000,9.071568,1.000000,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts
16672,bacteria:bc8d197b22a15c943a05,bacteria,76,200,18244,6932.720000,8.844008,0.974915,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts
15835,bacteria:b2e1f605564854e2b295,bacteria,70,200,19100,6685.000000,8.807621,0.970904,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts
3511,bacteria:28f6bcac134882921932,bacteria,78,200,16927,6601.530000,8.795057,0.969519,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts
8300,bacteria:5e57f2cbc058de6fa36c,bacteria,66,200,19345,6383.850000,8.761527,0.965823,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts
9610,bacteria:6ccde4b17efeee7c5ab7,bacteria,58,200,20529,5953.410000,8.691719,0.958128,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts
8483,bacteria:6044c66e7e71d401a3a9,bacteria,81,200,14529,5884.245000,8.680034,0.956839,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts
6785,bacteria:4d23437eb279fe4fb3af,bacteria,70,200,16581,5803.350000,8.666191,0.955313,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts
20222,bacteria:e365f618fb060a8ef6ae,bacteria,74,200,15644,5788.280000,8.663590,0.955027,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts
14212,bacteria:a07face9d26f567016b7,bacteria,87,200,13147,5718.945000,8.651540,0.953698,pathogenic-interest-index-v3.2-sampled,v2_migrated_counts


In [4]:
atomic_parquet(index, INDEX_PATH)
print(f'Wrote {len(index):,} rows to {INDEX_PATH}')


Wrote 142,201 rows to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version3/data/04_release/pathogenic_interest_index.parquet
